In [ ]:
# Scenario: AI-Assisted Email Workflow (Human-in-the-Loop)
# This workflow drafts a professional email using Groq API,
# pauses for human review, and then either sends or rejects it.

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict
import requests
from google.colab import userdata

 

class EmailState(TypedDict):
    task: str            # purpose/subject of the email
    recipient: str       # who the email is for
    tone: str            # professional / formal / polite / concise
    draft: str           # AI-generated email draft
    approved: bool       # human decision after review
    final_status: str    # SENT or REJECTED


 

def groq_call(prompt: str, model: str = "llama-3.1-8b-instant"):
    groq_api_key = userdata.get("groq_api_key")

    if not groq_api_key:
        raise ValueError(
            "Groq API key not found in Colab secrets. "
            "Please set the secret name exactly as 'groq_api_key'."
        )

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {groq_api_key}",
            "Content-Type": "application/json"
        },
        json={
            "model": model,
            "messages": [
                {"role": "user", "content": prompt}
            ],
            "temperature": 0.4
        }
    )

    if response.status_code != 200:
        try:
            error_details = response.json()
        except requests.exceptions.JSONDecodeError:
            error_details = response.text
        raise Exception(f"Groq API error (Status: {response.status_code}): {error_details}")

    response_json = response.json()

    if "choices" not in response_json or not response_json["choices"]:
        raise ValueError(f"Unexpected API response format: {response_json}")

    return response_json["choices"][0]["message"]["content"].strip()


 

def draft_email(state: EmailState):
    prompt = f"""
You are an AI email assistant.

Write a {state['tone']} professional email.

Task/Purpose: {state['task']}
Recipient: {state['recipient']}

Requirements:
- Keep the email clear and professional.
- Add a proper subject line at the top.
- Add greeting, body, and closing.
- Make the email realistic and workplace-ready.
"""

    draft = groq_call(prompt)

    print("\n📝 AI Draft Generated:\n")
    print(draft)

    return {"draft": draft}


 

def human_review(state: EmailState):
    print("\n📧 Draft ready for human review.\n")
    print(state["draft"])
    print("\n⏸ Workflow paused for approval.\n")
    return {}


 

def finalize_email(state: EmailState):
    if state.get("approved", False):
        print("\n✅ Draft approved by human reviewer.")
        print("📨 Email sent successfully.")
        return {"final_status": f"SENT: {state['task']}"}
    else:
        print("\n❌ Draft rejected by human reviewer.")
        print("📭 Email not sent.")
        return {"final_status": f"REJECTED: {state['task']}"}


 
g = StateGraph(EmailState)

g.add_node("draft", draft_email)
g.add_node("review", human_review)
g.add_node("finalize", finalize_email)

g.set_entry_point("draft")
g.add_edge("draft", "review")
g.add_edge("review", "finalize")
g.add_edge("finalize", END)


 

checkpointer = MemorySaver()

app = g.compile(
    checkpointer=checkpointer,
    interrupt_before=["review"]
)


 
if __name__ == "__main__":

    
    task_input = input("Enter email purpose/task: ")
    recipient_input = input("Enter recipient name or email role: ")
    tone_input = input("Enter tone (professional/formal/polite/concise): ")

    
    thread = {"configurable": {"thread_id": "email-assistant-1"}}

   
    app.invoke(
        {
            "task": task_input,
            "recipient": recipient_input,
            "tone": tone_input,
            "draft": "",
            "approved": False,
            "final_status": ""
        },
        thread
    )

    
    review_input = input("Approve this email? (yes/no): ").strip().lower()

    is_approved = True if review_input == "yes" else False
 
    result = app.invoke(
        {"approved": is_approved},
        thread
    )

     
    print("\n📌 Final Workflow Status:")
    print(result["final_status"])

Enter email purpose/task: prompt = f""" You are an expert corporate email assistant.  Write a {tone} professional email.  Details: - Purpose: {task} - Recipient: {recipient}  Instructions: - Add a clear subject line. - Start with a proper greeting. - Keep the tone professional and polite. - Write a concise but complete message. - Use clear paragraphs. - End with a professional closing.  Output format: Subject: <subject>  Email: <email body> """
Enter recipient name or email role: mAYANK
Enter tone (professional/formal/polite/concise): formal

📝 AI Draft Generated:

Subject: Request for Meeting to Discuss Upcoming Project Timeline

Email:

Dear Mayank,

I hope this email finds you well. I am writing to request a meeting with you to discuss the upcoming project timeline for the Smith account. As we approach the project's deadline, I believe it would be beneficial to review the current progress and make any necessary adjustments to ensure its timely completion.

During the meeting, we can

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict
import requests
from google.colab import userdata

 
class EmailState(TypedDict):
    task: str
    recipient: str
    tone: str
    draft: str
    approved: bool
    feedback: str
    final_status: str


 
def groq_call(prompt: str, model: str = "llama-3.1-8b-instant"):
    groq_api_key = userdata.get("groq_api_key")

    if not groq_api_key:
        raise ValueError(
            "Groq API key not found in Colab secrets. "
            "Please set the secret name exactly as 'groq_api_key'."
        )

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {groq_api_key}",
            "Content-Type": "application/json"
        },
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.4
        }
    )

    if response.status_code != 200:
        try:
            error_details = response.json()
        except requests.exceptions.JSONDecodeError:
            error_details = response.text
        raise Exception(f"Groq API error (Status: {response.status_code}): {error_details}")

    response_json = response.json()

    if "choices" not in response_json or not response_json["choices"]:
        raise ValueError(f"Unexpected API response format: {response_json}")

    return response_json["choices"][0]["message"]["content"].strip()

 
def draft_email(state: EmailState):
    
    if state.get("feedback", "").strip():
        prompt = f"""
You are an expert corporate email assistant.

Revise the following email draft based on reviewer feedback.

Task: {state['task']}
Recipient: {state['recipient']}
Tone: {state['tone']}

Current Draft:
{state['draft']}

Reviewer Feedback:
{state['feedback']}

Instructions:
- Improve the email using the feedback
- Keep it professional and polished
- Add a proper subject line
- Keep greeting, body, and closing
"""
    else:
        prompt = f"""
You are an expert corporate email assistant.

Write a {state['tone']} professional email.

Task: {state['task']}
Recipient: {state['recipient']}

Instructions:
- Add a proper subject line
- Keep the email clear and professional
- Include greeting, body, and closing
- Make it workplace ready
"""

    draft = groq_call(prompt)

    print("\n📝 Draft Email:\n")
    print(draft)

    return {"draft": draft}


 
def review_email(state: EmailState):
    print("\n📧 Email ready for human review:\n")
    print(state["draft"])
    print("\n⏸ Paused for review. Human can approve/reject and add feedback.\n")
    return {}

 
def send_email(state: EmailState):
    if state.get("approved", False):
        print("\n✅ Email approved and sent.")
        return {"final_status": f"SENT: {state['task']}"}
    else:
        print("\n❌ Email not approved.")
        return {"final_status": f"REJECTED: {state['task']}"}


 
g = StateGraph(EmailState)

g.add_node("draft", draft_email)
g.add_node("review", review_email)
g.add_node("send", send_email)

g.set_entry_point("draft")
g.add_edge("draft", "review")
g.add_edge("review", "send")
g.add_edge("send", END)


 
checkpointer = MemorySaver()

app = g.compile(
    checkpointer=checkpointer,
    interrupt_before=["review"]
)


 
if __name__ == "__main__":
    task_input = input("Enter email purpose/task: ")
    recipient_input = input("Enter recipient: ")
    tone_input = input("Enter tone (formal/professional/polite/concise): ")

    thread = {"configurable": {"thread_id": "email-hitl-1"}}

 
    app.invoke(
        {
            "task": task_input,
            "recipient": recipient_input,
            "tone": tone_input,
            "draft": "",
            "approved": False,
            "feedback": "",
            "final_status": ""
        },
        thread
    )

 
    decision = input("Approve email? (yes/no): ").strip().lower()
    feedback_input = input("Enter reviewer feedback (or press Enter to skip): ").strip()

    
    if decision == "no" and feedback_input:
        print("\n🔁 Revising draft based on feedback...\n")

        revised_result = app.invoke(
            {
                "approved": False,
                "feedback": feedback_input
            },
            thread
        )

        print("\n📌 Revised Draft Generated Above. Please review again.\n")

        second_decision = input("Approve revised email? (yes/no): ").strip().lower()

        final_result = app.invoke(
            {
                "approved": True if second_decision == "yes" else False
            },
            thread
        )

    else:
        final_result = app.invoke(
            {
                "approved": True if decision == "yes" else False,
                "feedback": feedback_input
            },
            thread
        )

    print("\nFinal Status:")
    print(final_result["final_status"])

Enter email purpose/task: write the email to mr sigh for give my money back
Enter recipient: singh@gmail.com
Enter tone (formal/professional/polite/concise): polite

📝 Draft Email:

Subject: Request for Refund of Recent Transaction

Dear Mr. Sigh,

I am writing to request a refund for a recent transaction that I believe was processed in error. I would greatly appreciate it if you could look into this matter and process a refund for the amount of [insert amount] as soon as possible.

If there is any additional information or documentation required from my end, please do not hesitate to reach out to me. I can be contacted at [insert your email address] or [insert your phone number].

Thank you for your prompt attention to this matter and I look forward to hearing from you soon.

Best regards,

[Your Name]
Approve email? (yes/no): no
Enter reviewer feedback (or press Enter to skip): add the money point of view that geve my 60000000 mone yck 

🔁 Revising draft based on feedback...


📝 Draf